In [2]:
pip install wbdata


Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for wbdata from https://files.pythonhosted.org/packages/f0/60/fa9661d0480d6f62642feeff217b86b60946cba8c7f04d35d6e342ef4309/wbdata-1.0.0-py3-none-any.whl.metadata
  Obtaining dependency information for backoff<3.0.0,>=2.2.1 from https://files.pythonhosted.org/packages/df/73/b6e24bd22e6720ca8ee9a85a0c4a2971af8497d8f3193fa05390cbd46e09/backoff-2.2.1-py3-none-any.whl.metadata
  Obtaining dependency information for cachetools<6.0.0,>=5.3.2 from https://files.pythonhosted.org/packages/72/76/20fa66124dbe6be5cafeb312ece67de6b61dd91a0247d1ea13db4ebb33c2/cachetools-5.5.2-py3-none-any.whl.metadata
  Obtaining dependency information for dateparser<2.0.0,>=1.2.0 from https://files.pythonhosted.org/packages/87/22/f020c047ae1346613db9322638186468238bcfa8849b4668a22b97faad65/dateparser-1.2.2-py3-none-any.whl.metadata
  Obtaining dependency information for shelved-cache<0.4.0,>=0.3.1 from ht

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires PyYAML==6.0.1, but you have pyyaml 6.0.2 which is incompatible.
conda-repo-cli 1.0.75 requires requests==2.31.0, but you have requests 2.32.4 which is incompatible.


In [6]:
pip install fredapi


Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for fredapi from https://files.pythonhosted.org/packages/73/64/1db43417cf7ed430f104a347126b5260a1724ee9a1b7d0b1622262c9c4df/fredapi-0.5.2-py3-none-any.whl.metadata
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pycountry


Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for pycountry from https://files.pythonhosted.org/packages/b1/ec/1fb891d8a2660716aadb2143235481d15ed1cbfe3ad669194690b0604492/pycountry-24.6.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
    --------------------------------------- 0.1/6.3 MB 1.3 MB/s eta 0:00:05
   - -------------------------------------- 0.3/6.3 MB 2.4 MB/s eta 0:00:03
   ---- ----------------------------------- 0.7/6.3 MB 4.1 MB/s eta 0:00:02
   -------- ------------------------------- 1.4/6.3 MB 6.7 MB/s eta 0:00:01
   -------------- ------------------------- 2.3/6.3 MB 9.0 MB/s eta 0:00:01
   ------------------------ --------------- 3.9/6.3 MB 13.0 MB/s eta 0:00:01
   --------------------------------- ------ 5.2/6.3 MB 15.2 MB/s eta 0:00:01
   ---------------------------------------  6.

In [11]:
pip install wbgapi wbdata fredapi pandas numpy requests tqdm python-dateutil


Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for wbgapi from https://files.pythonhosted.org/packages/00/12/224030af4886e119a3d03b709f7130e9601a4d15332e1d6a35671b25a4de/wbgapi-1.0.12-py3-none-any.whl.metadata
Note: you may need to restart the kernel to use updated packages.


In [12]:
# -------------------------------------------
# Sovereign Risk Panel Builder (2025 rewrite)
# -------------------------------------------
# Default: World Bank via wbgapi (modern & stable)
# Fallback: wbdata >= 1.0 (params fixed)
# WGI: provide CSV "data_out/wgi_full.csv" (Aggregate Indicators)
# FRED: monthly -> annual mean; then broadcast to panel
#
# Core outputs:
#   data_out/panel_full_pre_filter.csv
#   data_out/panel_final.csv
# -------------------------------------------

import os, warnings
from datetime import datetime
import pandas as pd
import numpy as np
from tqdm import tqdm

warnings.filterwarnings("ignore")

# -----------------------------
# Config
# -----------------------------
START_YEAR, END_YEAR = 1980, 2022
YEARS = list(range(START_YEAR, END_YEAR + 1))
OUT_DIR = "data_out"
os.makedirs(OUT_DIR, exist_ok=True)

# World Bank indicators
WDI_INDICATORS = {
    "GC.DOD.TOTL.GD.ZS": "debt_gdp",      # Central gov debt (% GDP)
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",    # GDP growth (annual %)
    "FP.CPI.TOTL.ZG":    "inflation_cpi", # Inflation, CPI (annual %)
    "NE.TRD.GNFS.ZS":    "trade_gdp",     # Trade (% of GDP)
    "FI.RES.TOTL.MO":    "reserves_months" # Total reserves (months of imports)
}

# WGI mapping for wide pivot
WGI_EXPECTED = {
    "Voice and Accountability": "wgi_voice",
    "Political Stability and Absence of Violence/Terrorism": "wgi_stability",
    "Government Effectiveness": "wgi_effectiveness",
    "Regulatory Quality": "wgi_regulatory",
    "Rule of Law": "wgi_ruleoflaw",
    "Control of Corruption": "wgi_corruption",
}

# FRED series (monthly)
FRED_SERIES = {
    "FEDFUNDS": "fed_funds_rate",   # Effective Fed Funds Rate
    "DCOILBRENTEU": "brent_usd_bbl" # Brent spot price (USD/bbl)
}

# -----------------------------
# Helpers
# -----------------------------
def ensure_years(df):
    """Ensure every country has the full YEAR range."""
    out = []
    base = pd.DataFrame({"year": YEARS})
    for c, g in df.groupby("country", sort=True):
        gg = base.merge(g, on="year", how="left")
        gg.insert(0, "country", c)
        out.append(gg)
    return pd.concat(out, ignore_index=True)

def ffill_within_country(df, cols):
    return (df.sort_values(["country","year"])
              .groupby("country", group_keys=False)
              .apply(lambda g: g.assign(**{c: g[c].ffill() for c in cols})))

def _mean_impute_short_gaps(series, max_gap=3):
    s = series.copy()
    if s.notna().sum() == 0: return s
    m = s.mean(skipna=True)
    if np.isnan(m): return s
    is_na = s.isna().astype(int)
    groups = (is_na.diff().ne(0)).cumsum()
    for gid in groups[is_na.eq(1)].unique():
        idx = s.index[groups == gid]
        if len(idx) <= max_gap:
            s.loc[idx] = m
    return s

def mean_impute_short_by_group(df, cols, max_gap=3):
    def _g(g):
        for c in cols:
            g[c] = _mean_impute_short_gaps(g[c], max_gap=max_gap)
        return g
    return df.groupby("country", group_keys=False).apply(_g)

def annualize_monthly(df_m, date_col, value_col, out_name):
    df_m["year"] = pd.to_datetime(df_m[date_col]).dt.year
    df_y = (df_m.groupby(["country","year"], as_index=False)[value_col]
                  .mean()
                  .rename(columns={value_col: out_name}))
    return df_y[df_y["year"].between(START_YEAR, END_YEAR)]

# -----------------------------
# 1) World Bank via wbgapi (preferred)
# -----------------------------
def fetch_wdi_wbgapi():
    import wbgapi as wb
    codes = list(WDI_INDICATORS.keys())
    economies = list(wb.economy.list(skipAggs=True))  # no aggregates
    wdi = wb.data.DataFrame(
        codes,
        economy=economies,
        time=range(START_YEAR, END_YEAR+1),
        numericTimeKeys=True,
        labels=True,
        columns="series"
    ).reset_index()  # columns: economy, Time, series, value
    wdi["series"] = wdi["series"].map(WDI_INDICATORS)
    wdi = (wdi.pivot_table(index=["economy","Time"], columns="series", values="value", aggfunc="mean")
             .reset_index()
             .rename(columns={"economy":"country", "Time":"year"}))
    return wdi

# -----------------------------
# 1b) Fallback: World Bank via wbdata (with 1.0 arg changes)
# -----------------------------
def fetch_wdi_wbdata():
    import wbdata
    # countries (skip aggregates)
    all_cty = wbdata.get_countries()
    def _ok(c):
        r = c.get("region", {})
        return (r.get("id") != "NA") and (r.get("value") != "Aggregates")
    countries = [c.get("id","") for c in all_cty if len(c.get("id",""))==3 and _ok(c)]
    # data (note: data_date->date, convert_dates->parse_dates in v1.0+)
    wdf = wbdata.get_dataframe(
        WDI_INDICATORS,
        country=countries,
        date=(str(START_YEAR), str(END_YEAR)),
        parse_dates=True
    ).reset_index().rename(columns={"date":"date"})
    wdf["year"] = pd.to_datetime(wdf["date"]).dt.year
    wdf = wdf.drop(columns=["date"])
    # tidy to full panel years
    wdi = (wdf.groupby(["country","year"], as_index=False)
             .mean(numeric_only=True))
    return ensure_years(wdi)

# -----------------------------
# 2) WGI: read CSV saved as data_out/wgi_full.csv
#     (download Aggregate Indicators from the official WGI/DataBank)
# -----------------------------
def load_wgi_csv(path):
    if not os.path.exists(path):
        print("WGI CSV not found. Download the 'Aggregate Indicators' CSV and save as:", path)
        return pd.DataFrame(columns=["country","year"] + list(WGI_EXPECTED.values()))
    raw = pd.read_csv(path, low_memory=False)
    raw = raw.rename(columns={c: c.strip() for c in raw.columns})
    ctry_cols = [c for c in ["Country/Territory","Country Name","Country"] if c in raw.columns]
    code_cols = [c for c in ["Code","Country Code","ISO Code"] if c in raw.columns]
    series_col = "Series" if "Series" in raw.columns else ("Indicator" if "Indicator" in raw.columns else None)
    value_col  = "Estimate" if "Estimate" in raw.columns else ("Value" if "Value" in raw.columns else None)
    if not ctry_cols or not code_cols or not series_col or not value_col:
        raise ValueError("Unrecognised WGI CSV format. Use the official 'Aggregate Indicators' export.")
    raw = raw[[ctry_cols[0], code_cols[0], series_col, "Year", value_col]].rename(
        columns={ctry_cols[0]:"country_name", code_cols[0]:"country", series_col:"series", "Year":"year", value_col:"value"}
    )
    raw = raw[raw["year"].between(START_YEAR, END_YEAR)]
    keep = raw[raw["series"].isin(WGI_EXPECTED.keys())].copy()
